# Idealista Barcelona Detail Batches

This notebook only processes the existing URL checkpoint:

`data/idealista_barcelona_sale_urls.csv`

It does **not** scrape search pages. It gently visits listing detail pages in small resumable batches and writes:

`data/idealista_barcelona_sale_properties_details.csv`

In [11]:
import csv
import hashlib
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

## Controls

For gentle scraping, keep `BATCH_SIZE` small and delays fairly wide. Rerun later to continue; already completed URLs are skipped.

In [12]:
OUT_DIR = Path("data")
URLS_CSV = OUT_DIR / "idealista_barcelona_sale_urls.csv"
OUTPUT_CSV = OUT_DIR / "idealista_barcelona_sale_properties_details.csv"
DETAIL_CACHE_DIR = OUT_DIR / "html_cache" / "detail_pages"
DETAIL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Browser controls
HEADLESS = False
USE_CHROME_PROFILE = True
CHROME_PROFILE_DIR = OUT_DIR / "chrome_profile"  # keeps cookies/CAPTCHA clearance between runs
CHROMEDRIVER_PATH = None   # set to r"C:\\path\\to\\chromedriver.exe" if needed
USE_WEBDRIVER_MANAGER = True

# Gentle batch controls
BATCH_SIZE = 50            # number of new listing pages to process in this run
MIN_DELAY = 5
MAX_DELAY = 12
LONG_BREAK_EVERY = 10      # take a longer pause every N listings; set None to disable
LONG_BREAK_MIN = 10
LONG_BREAK_MAX = 20
LOG_EVERY = 1

# Cache controls
USE_HTML_CACHE = True
REFRESH_HTML_CACHE = False

# Block handling
WAIT_ON_BLOCK = True
MANUAL_UNBLOCK_CONFIRM = True
BLOCK_WAIT_SECONDS = 300
STOP_ON_HARD_BLOCK = True

COLUMNS = [
    "propertyCode",
    "Link",
    "district",
    "neighborhood",
    "price",
    "size",
    "bed",
    "br",
    "floor",
    "address",
    "latitude",
    "longitude",
    "x",
    "y",
    "url",
    "description",
    "scraped_at",
]

In [13]:
def now_utc_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def clean_text(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value or None


def property_code(url):
    match = re.search(r"/inmueble/(\d+)/", str(url))
    return match.group(1) if match else None


def listing_key(row):
    code = row.get("propertyCode") if isinstance(row, dict) else None
    url = row.get("url") if isinstance(row, dict) else None
    return str(code) if code not in (None, "", "nan") else str(url)


def as_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)) and not pd.isna(value):
        return value
    match = re.search(r"-?\d+(?:[.,]\d+)?", str(value).replace(".", ""))
    if not match:
        return None
    text = match.group(0).replace(",", ".")
    number = float(text)
    return int(number) if number.is_integer() else number


def polite_sleep(index=None):
    delay = random.uniform(MIN_DELAY, MAX_DELAY)
    log(f"Sleeping {delay:.1f}s")
    time.sleep(delay)
    if LONG_BREAK_EVERY and index and index % LONG_BREAK_EVERY == 0:
        long_delay = random.uniform(LONG_BREAK_MIN, LONG_BREAK_MAX)
        log(f"Long break after {index} listings: {long_delay:.1f}s")
        time.sleep(long_delay)

In [14]:
def setup_driver():
    log("Starting Chrome WebDriver")
    options = Options()
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--lang=en-US,en")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if USE_CHROME_PROFILE:
        CHROME_PROFILE_DIR.mkdir(parents=True, exist_ok=True)
        options.add_argument(f"--user-data-dir={CHROME_PROFILE_DIR.resolve()}")
        log(f"Using persistent Chrome profile: {CHROME_PROFILE_DIR.resolve()}")
    if HEADLESS:
        options.add_argument("--headless=new")

    if CHROMEDRIVER_PATH:
        log(f"Using explicit ChromeDriver path: {CHROMEDRIVER_PATH}")
        driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH), options=options)
    elif USE_WEBDRIVER_MANAGER:
        log("Resolving ChromeDriver with webdriver-manager")
        driver_path = ChromeDriverManager().install()
        log(f"ChromeDriver resolved: {driver_path}")
        driver = webdriver.Chrome(service=Service(driver_path), options=options)
    else:
        log("Resolving ChromeDriver with Selenium Manager")
        driver = webdriver.Chrome(options=options)

    driver.set_page_load_timeout(45)
    log("Chrome WebDriver ready")
    return driver


def accept_cookies(driver):
    for label in ["Accept all", "Accept", "Agree", "Aceptar todas", "Aceptar"]:
        try:
            buttons = driver.find_elements(By.XPATH, f"//button[contains(normalize-space(.), '{label}')]")
            for button in buttons:
                if button.is_displayed() and button.is_enabled():
                    log("Accepting cookie banner")
                    button.click()
                    time.sleep(1)
                    return
        except WebDriverException:
            pass


def safe_get(driver, url):
    try:
        log(f"Loading: {url}")
        driver.get(url)
        return True
    except TimeoutException:
        log(f"Timeout while loading, keeping partial page: {url}")
        try:
            driver.execute_script("window.stop();")
        except WebDriverException:
            pass
        return True
    except WebDriverException as exc:
        log(f"Could not load {url}: {exc}")
        return False

In [15]:
def visible_page_text(driver):
    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
        title = (driver.title or "").lower()
        current_url = (driver.current_url or "").lower()
        return " ".join([title, current_url, body_text])
    except WebDriverException as exc:
        log(f"Could not inspect visible page text: {exc}")
        return ""


def hard_blocked(driver):
    text = visible_page_text(driver)
    hard_markers = [
        "se ha detectado un uso indebido",
        "el acceso se ha bloqueado",
        "póngase en contacto con el servicio de asistencia",
        "pongase en contacto con el servicio de asistencia",
    ]
    return any(marker in text for marker in hard_markers)


def blocked(driver):
    text = visible_page_text(driver)
    hard_markers = ["unusual traffic", "verify you are human", "access denied", "checking your browser"]
    if any(marker in text for marker in hard_markers):
        return True
    has_listing_content = "property for sale" in text or "houses and flats" in text or "m²" in text or "eur" in text
    return "captcha" in text and not has_listing_content


def wait_for_manual_unblock(driver, url):
    if STOP_ON_HARD_BLOCK and hard_blocked(driver):
        raise RuntimeError("Idealista is showing a hard access block. Stop for now; completed rows are already checkpointed.")
    if not WAIT_ON_BLOCK:
        return False

    log("Idealista appears to be showing a block/CAPTCHA page")
    if MANUAL_UNBLOCK_CONFIRM:
        input("After the Idealista page looks normal in Chrome, press Enter here to continue: ")
        if STOP_ON_HARD_BLOCK and hard_blocked(driver):
            raise RuntimeError("Idealista is showing a hard access block. Stop for now; completed rows are already checkpointed.")
        if not blocked(driver):
            log("Manual unblock confirmed; continuing")
            return True

    deadline = time.time() + BLOCK_WAIT_SECONDS
    while time.time() < deadline:
        if STOP_ON_HARD_BLOCK and hard_blocked(driver):
            raise RuntimeError("Idealista is showing a hard access block. Stop for now; completed rows are already checkpointed.")
        if not blocked(driver):
            log("Block/CAPTCHA appears cleared; continuing")
            return True
        log("Still blocked; waiting 10 seconds before checking again")
        time.sleep(10)
    return False

In [16]:
def cache_path(url):
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    code = property_code(url)
    stem = f"{code}_{digest}" if code else digest
    return DETAIL_CACHE_DIR / f"{stem}.html"


def load_cached_html(url):
    path = cache_path(url)
    if USE_HTML_CACHE and not REFRESH_HTML_CACHE and path.exists():
        log(f"Cache hit: {path}")
        return path.read_text(encoding="utf-8")
    return None


def save_cached_html(url, html):
    if not USE_HTML_CACHE or not html:
        return None
    path = cache_path(url)
    path.write_text(html, encoding="utf-8")
    log(f"Cached HTML: {path}")
    return path


def current_html(driver):
    try:
        return driver.page_source
    except WebDriverException as exc:
        log(f"driver.page_source failed; trying JavaScript HTML read: {exc}")
        try:
            return driver.execute_script("return document.documentElement.outerHTML")
        except WebDriverException as js_exc:
            log(f"JavaScript HTML read also failed: {js_exc}")
            return None


def get_detail_html(driver, url):
    cached = load_cached_html(url)
    if cached is not None:
        return cached

    if not safe_get(driver, url):
        return None
    accept_cookies(driver)
    if STOP_ON_HARD_BLOCK and hard_blocked(driver):
        raise RuntimeError("Idealista is showing a hard access block. Stop for now; completed rows are already checkpointed.")
    if blocked(driver):
        if not wait_for_manual_unblock(driver, url):
            raise RuntimeError("Idealista is still showing a block/CAPTCHA. Stop and retry later.")

    try:
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "body")))
    except TimeoutException:
        log("Timed out waiting for body; trying to read current HTML anyway")

    html = current_html(driver)
    if html:
        save_cached_html(url, html)
    return html

## Parsing Helpers

In [17]:
def json_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from json_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from json_objects(child)


def load_jsonld(soup):
    objects = []
    for script in soup.select("script[type='application/ld+json']"):
        raw = script.string or script.get_text()
        if not raw:
            continue
        try:
            objects.extend(list(json_objects(json.loads(raw))))
        except Exception:
            continue
    return objects


def first_schema_value(objects, keys):
    keys = {k.lower() for k in keys}
    for obj in objects:
        for key, value in obj.items():
            if str(key).lower() in keys and value not in (None, "", []):
                return value
    return None


def schema_address(objects):
    value = first_schema_value(objects, ["address"])
    if isinstance(value, dict):
        parts = [value.get(k) for k in ["streetAddress", "addressLocality", "addressRegion", "postalCode"]]
        return clean_text(", ".join(str(x) for x in parts if x))
    return clean_text(value)


def schema_geo(objects):
    for obj in objects:
        if "latitude" in obj and "longitude" in obj:
            return obj.get("latitude"), obj.get("longitude")
        geo = obj.get("geo")
        if isinstance(geo, dict) and "latitude" in geo and "longitude" in geo:
            return geo.get("latitude"), geo.get("longitude")
    return None, None


def regex_geo(html):
    patterns = [
        r'"latitude"\s*:\s*([\-\d.]+)\s*,\s*"longitude"\s*:\s*([\-\d.]+)',
        r'"longitude"\s*:\s*([\-\d.]+)\s*,\s*"latitude"\s*:\s*([\-\d.]+)',
        r'lat(?:itude)?["\']?\s*[:=]\s*["\']?([\-\d.]+).*?lon(?:gitude)?["\']?\s*[:=]\s*["\']?([\-\d.]+)',
    ]
    for i, pattern in enumerate(patterns):
        match = re.search(pattern, html, flags=re.I | re.S)
        if match and i == 1:
            return match.group(2), match.group(1)
        if match:
            return match.group(1), match.group(2)
    return None, None


def visible_text(soup, selectors):
    for selector in selectors:
        node = soup.select_one(selector)
        if node:
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                return text
    return None


def detail_text(soup, fallback=None):
    chunks = [x.get_text(" ", strip=True) for x in soup.select(".details-property_features li, .info-features span, .details-property-feature-one, .details-property-feature-two, .item-detail")]
    return clean_text(" | ".join(chunks)) or fallback


def parse_features(text):
    text = clean_text(text) or ""
    out = {"size": None, "bed": None, "br": None, "floor": None}
    m = re.search(r"([\d.,]+)\s*m[²2]", text, flags=re.I)
    if m:
        out["size"] = as_number(m.group(1))
    m = re.search(r"(\d+)\s*bed", text, flags=re.I)
    if m:
        out["bed"] = int(m.group(1))
    m = re.search(r"(\d+)\s*bath", text, flags=re.I)
    if m:
        out["br"] = int(m.group(1))
    for pattern in [r"(\d+)(?:st|nd|rd|th)?\s*floor", r"floor\s*(\d+)", r"(ground floor|basement|semi-basement|mezzanine|top floor)"]:
        m = re.search(pattern, text, flags=re.I)
        if m:
            out["floor"] = m.group(1).lower()
            break
    return out


def parse_location_from_address(address):
    parts = [clean_text(x) for x in str(address or "").split(",")]
    parts = [x for x in parts if x]
    district = None
    neighborhood = None
    if len(parts) >= 3:
        neighborhood = parts[-2]
        district = parts[-1].replace("Barcelona", "").strip() or None
    elif len(parts) == 2:
        neighborhood = parts[-1]
    return district, neighborhood

In [18]:
def parse_listing_detail(html, url, source_row):
    soup = BeautifulSoup(html, "lxml")
    objects = load_jsonld(soup)
    features = parse_features(detail_text(soup, source_row.get("details_search")))

    schema_lat, schema_lon = schema_geo(objects)
    regex_lat, regex_lon = regex_geo(html)
    latitude = schema_lat or regex_lat
    longitude = schema_lon or regex_lon

    address = (
        schema_address(objects)
        or visible_text(soup, ["span.main-info__title-main", ".main-info__title-main", "h1"])
        or source_row.get("address_search")
    )
    district, neighborhood = parse_location_from_address(address)

    price_value = first_schema_value(objects, ["price"])
    price = clean_text(price_value) or visible_text(soup, ["span.info-data-price", ".info-data-price", "span.item-price", ".item-price"])

    description = (
        clean_text(first_schema_value(objects, ["description"]))
        or visible_text(soup, ["div.comment", ".adCommentsLanguage", "#details .comment", "[class*='description']"])
        or source_row.get("description_search")
    )

    size_schema = first_schema_value(objects, ["floorSize", "size", "area"])
    if isinstance(size_schema, dict):
        size_schema = size_schema.get("value") or size_schema.get("amount")

    return {
        "propertyCode": source_row.get("propertyCode") or property_code(url),
        "Link": "LINK",
        "district": district,
        "neighborhood": neighborhood,
        "price": price or source_row.get("price_search"),
        "size": as_number(size_schema) or features["size"],
        "bed": as_number(first_schema_value(objects, ["numberOfBedrooms", "numberOfRooms"])) or features["bed"],
        "br": as_number(first_schema_value(objects, ["numberOfBathroomsTotal", "numberOfBathrooms"])) or features["br"],
        "floor": clean_text(first_schema_value(objects, ["floorLevel", "floor"])) or features["floor"],
        "address": address,
        "latitude": latitude,
        "longitude": longitude,
        "x": longitude,
        "y": latitude,
        "url": url,
        "description": description,
        "scraped_at": now_utc_iso(),
    }


def summarize_detail_row(row):
    fields = ["price", "size", "bed", "br", "floor", "address", "latitude", "longitude", "description"]
    present = [name for name in fields if row.get(name) not in (None, "")]
    missing = [name for name in fields if row.get(name) in (None, "")]
    return f"present={present}; missing={missing}"

## Load Checkpoints

In [19]:
def load_url_rows():
    if not URLS_CSV.exists():
        raise FileNotFoundError(f"Missing URL checkpoint: {URLS_CSV}")
    df = pd.read_csv(URLS_CSV)
    if "url" not in df.columns:
        raise ValueError(f"URL checkpoint must have a 'url' column: {URLS_CSV}")
    if "propertyCode" not in df.columns:
        df["propertyCode"] = df["url"].map(property_code)
    df["_key"] = df.apply(lambda row: listing_key(row.to_dict()), axis=1)
    df = df.dropna(subset=["url"]).drop_duplicates("_key", keep="last")
    log(f"Loaded {len(df):,} unique URLs from {URLS_CSV}")
    return df


def load_existing_details():
    if OUTPUT_CSV.exists():
        df = pd.read_csv(OUTPUT_CSV)
        if "propertyCode" not in df.columns:
            df["propertyCode"] = df["url"].map(property_code)
        df["_key"] = df.apply(lambda row: listing_key(row.to_dict()), axis=1)
        done_keys = set(df["_key"].dropna().astype(str))
        log(f"Loaded {len(df):,} existing detail rows from {OUTPUT_CSV}")
        return df.reindex(columns=COLUMNS + ["_key"]), done_keys
    log("No existing detail output found; starting detail output from scratch")
    return pd.DataFrame(columns=COLUMNS + ["_key"]), set()


urls_df = load_url_rows()
existing_df, done_keys = load_existing_details()
todo_df = urls_df[~urls_df["_key"].astype(str).isin(done_keys)].copy()
if BATCH_SIZE:
    todo_df = todo_df.head(BATCH_SIZE)
log(f"Todo this run: {len(todo_df):,}; already done: {len(done_keys):,}")
todo_df.head()

[19:14:16] Loaded 10,614 unique URLs from data\idealista_barcelona_sale_urls.csv
[19:14:16] Loaded 13 existing detail rows from data\idealista_barcelona_sale_properties_details.csv
[19:14:16] Todo this run: 50; already done: 13


,propertyCode,url,address_search,price_search,details_search,description_search,source_page,scraped_at,seed_url,seed_label,_key
13,110437931,https://www.idealista.com/inmueble/110437931/,Flat / apartment in Calle President Lluís Comp...,NaN,NaN,NaN,https://www.idealista.com/en/venta-viviendas/b...,2026-04-17T15:35:12+00:00,NaN,NaN,110437931
14,109853125,https://www.idealista.com/inmueble/109853125/,"Flat / apartment in Pasaje Salut, 96",NaN,NaN,NaN,https://www.idealista.com/en/venta-viviendas/b...,2026-04-17T15:35:12+00:00,NaN,NaN,109853125
15,110897278,https://www.idealista.com/inmueble/110897278/,"Flat / apartment in Calle Maternitat D’elna, 3",NaN,NaN,NaN,https://www.idealista.com/en/venta-viviendas/b...,2026-04-17T15:35:12+00:00,NaN,NaN,110897278
16,110971923,https://www.idealista.com/inmueble/110971923/,"Flat / apartment in Riera De La Creu 54, 56",NaN,NaN,NaN,https://www.idealista.com/en/venta-viviendas/b...,2026-04-17T15:35:12+00:00,NaN,NaN,110971923
17,110975132,https://www.idealista.com/inmueble/110975132/,"Detached house in Avenida Mas Fuster, 144",NaN,NaN,NaN,https://www.idealista.com/en/venta-viviendas/b...,2026-04-17T15:35:12+00:00,NaN,NaN,110975132


## Run Detail Batch

In [20]:
def save_details(rows):
    df = pd.DataFrame(rows).reindex(columns=COLUMNS + ["_key"])
    df = df.drop_duplicates("_key", keep="last")
    df.reindex(columns=COLUMNS).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
    return df


driver = setup_driver()
rows = existing_df.to_dict("records")

try:
    for index, source_row in enumerate(tqdm(todo_df.to_dict("records"), desc="Detail batch"), start=1):
        url = source_row["url"]
        code = source_row.get("propertyCode") or property_code(url)
        key = listing_key({"propertyCode": code, "url": url})
        if LOG_EVERY and (index == 1 or index % LOG_EVERY == 0):
            log(f"Detail {index}/{len(todo_df)} start: {code} | {url}")

        html = get_detail_html(driver, url)
        if not html:
            log(f"No HTML returned; skipping for now: {url}")
            polite_sleep(index)
            continue

        parsed = parse_listing_detail(html, url, source_row)
        parsed["_key"] = key
        rows.append(parsed)
        current_df = save_details(rows)

        if LOG_EVERY and (index == 1 or index % LOG_EVERY == 0):
            log(f"Detail {index}/{len(todo_df)} saved: {code}; {summarize_detail_row(parsed)}")
            log(f"Output rows now: {len(current_df):,} -> {OUTPUT_CSV}")

        polite_sleep(index)
finally:
    try:
        driver.quit()
        log("Chrome WebDriver closed")
    except Exception:
        pass

properties_df = pd.read_csv(OUTPUT_CSV) if OUTPUT_CSV.exists() else pd.DataFrame(columns=COLUMNS)
properties_df.head(), properties_df.shape

[19:14:16] Starting Chrome WebDriver
[19:14:16] Using persistent Chrome profile: C:\Users\sffra\Downloads\BSE 2025-2026\scrape-idealista\data\chrome_profile
[19:14:16] Resolving ChromeDriver with webdriver-manager
[19:14:18] ChromeDriver resolved: C:\Users\sffra\.wdm\drivers\chromedriver\win64\147.0.7727.57\chromedriver-win32/chromedriver.exe
[19:14:20] Chrome WebDriver ready


Detail batch:   0%|          | 0/50 [00:00<?, ?it/s]

[19:14:20] Detail 1/50 start: 110437931 | https://www.idealista.com/inmueble/110437931/
[19:14:20] Cache hit: data\html_cache\detail_pages\110437931_5ae875ed184f803b.html
[19:14:20] Detail 1/50 saved: 110437931; present=['price', 'size', 'address', 'description']; missing=['bed', 'br', 'floor', 'latitude', 'longitude']
[19:14:20] Output rows now: 14 -> data\idealista_barcelona_sale_properties_details.csv
[19:14:20] Sleeping 8.5s


Detail batch:   2%|▏         | 1/50 [00:08<07:06,  8.71s/it]

[19:14:28] Detail 2/50 start: 109853125 | https://www.idealista.com/inmueble/109853125/
[19:14:28] Cache hit: data\html_cache\detail_pages\109853125_df9bb177f28f60df.html
[19:14:28] Detail 2/50 saved: 109853125; present=['price', 'size', 'address', 'description']; missing=['bed', 'br', 'floor', 'latitude', 'longitude']
[19:14:28] Output rows now: 15 -> data\idealista_barcelona_sale_properties_details.csv
[19:14:28] Sleeping 9.3s


...

Detail batch:  16%|█▌        | 8/50 [01:08<07:08, 10.21s/it]

[19:15:28] Detail 9/50 start: 110454464 | https://www.idealista.com/inmueble/110454464/
[19:15:28] Loading: https://www.idealista.com/inmueble/110454464/
[19:15:29] Could not inspect visible page text: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=147.0.7727.101)
Stacktrace:
	chromedriver!GetHandleVerifier [0xb9c733+10b73]
	chromedriver!GetHandleVerifier [0xb9c864+10ca4]
	chromedriver!(No symbol) [0x972100]
	chromedriver!(No symbol) [0x950c23]
	chromedriver!(No symbol) [0x9e445b]
	chromedriver!(No symbol) [0x9fac29]
	chromedriver!(No symbol) [0x9dd9b6]
	chromedriver!(No symbol) [0x9b0339]
	chromedriver!(No symbol) [0x9b10f4]
	chromedriver!GetHandleVerifier [0xdffe04+274244]
	chromedriver!GetHandleVerifier [0xdfb459+26f899]
	chromedriver!GetHandleVerifier [0xe19bd5+28e015]
	chromedriver!GetHandleVerifier [0xbb7188+2b5c8]
	chromedriver!GetHandleVerifier [0xbbf23d+3367d]
	chromedriver!GetHandleVerifier [0xba5138+19578]

Detail batch:  16%|█▌        | 8/50 [01:09<06:03,  8.66s/it]


[19:15:31] Chrome WebDriver closed


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=147.0.7727.101)
Stacktrace:
	chromedriver!GetHandleVerifier [0xb9c733+10b73]
	chromedriver!GetHandleVerifier [0xb9c864+10ca4]
	chromedriver!(No symbol) [0x972100]
	chromedriver!(No symbol) [0x950c23]
	chromedriver!(No symbol) [0x9e445b]
	chromedriver!(No symbol) [0x9fac29]
	chromedriver!(No symbol) [0x9dd9b6]
	chromedriver!(No symbol) [0x9b0339]
	chromedriver!(No symbol) [0x9b10f4]
	chromedriver!GetHandleVerifier [0xdffe04+274244]
	chromedriver!GetHandleVerifier [0xdfb459+26f899]
	chromedriver!GetHandleVerifier [0xe19bd5+28e015]
	chromedriver!GetHandleVerifier [0xbb7188+2b5c8]
	chromedriver!GetHandleVerifier [0xbbf23d+3367d]
	chromedriver!GetHandleVerifier [0xba5138+19578]
	chromedriver!GetHandleVerifier [0xba5302+19742]
	chromedriver!GetHandleVerifier [0xb8e48f+28cf]
	KERNEL32!BaseThreadInitThunk [0x745d5d49+19]
	ntdll!RtlInitializeExceptionChain 

## Quality Check

In [ ]:
properties_df = pd.read_csv(OUTPUT_CSV)
display(properties_df.head())
display(properties_df.isna().mean().sort_values(ascending=False).to_frame("missing_share"))
print(f"Rows: {len(properties_df):,}")
print(f"Unique property codes: {properties_df['propertyCode'].nunique():,}")
print(f"Rows with x/y: {properties_df[['x', 'y']].notna().all(axis=1).sum():,}")
print(f"Output: {OUTPUT_CSV.resolve()}")